# Discriminant Classifiers

Using discriminant classifiers / Naive Bayes to classify our data.

In [1]:
import pandas as pd

In [2]:
learn_data = pd.read_csv("minimal_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,48,1.504077,5.641907,4.304065,2.4,0.52,0.511111,0,0
1,39,0.641854,5.192957,4.127134,4.3,1.38,0.473684,0,0
2,23,0.000000,5.356586,4.382027,3.1,1.00,0.300000,0,0
3,42,-0.356675,5.023881,4.394449,3.2,1.06,0.285714,1,0
4,54,3.117950,6.324359,3.610918,3.4,0.80,0.504425,1,0


In [3]:
learn_data.isna().value_counts()

Age    TB     Alkphos  Sgot   ALB    AR     BilRatio  Female  Target
False  False  False    False  False  False  False     False   False     449
Name: count, dtype: int64

In [4]:
X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

## Metrics

In [5]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear Discriminant Classifier

From PCA analysis, we know that our data can be separated in two clouds of sick and healthy patients respectively. A LD classifier might work well, but we know that the clouds may overlap. Also, their covariance matrices are clearly different. We might need to use a Quadratic Discriminant Classifier instead.

In [6]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.33, random_state = 42)

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
lda_model.fit(X_train, y_train)

print('Priors:', lda_model.priors_)

Priors: [0.5 0.5]


In [7]:
confusion(np.array(y_train), pd.Series(lda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	72	12
	0	82	134
Accuracy: 68.67%


In [8]:
confusion(np.array(y_val), pd.Series(lda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	31	11
	0	37	70
Accuracy: 67.79%


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

LDA_pipeline = Pipeline([('scaler', StandardScaler()), ('LDA', LinearDiscriminantAnalysis())])

n = 10
priors = [(p / n, 1 - p / n) for p in range(1, n)]

LDA_search = GridSearchCV(estimator = LDA_pipeline,
                          param_grid = {'LDA__priors' : priors},
                          scoring = 'f1_macro',
                          cv = 5)
LDA_search.fit(X, y)
LDA_search.best_params_

{'LDA__priors': (0.5, 0.5)}

In [10]:
from sklearn.model_selection import cross_validate

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
lda_pipeline = Pipeline([('scaler', StandardScaler()), ('LDA', lda_model)])

cross_val_results = pd.DataFrame(cross_validate(lda_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["LDA", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.648262,0.714125,0.674174,0.661648


## Quadratic Discrimant Classifier

We now use a QDA classifier. We see that the problem is preserved: the "sick" class overlaps too much with the healthy class and it

In [11]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5))
_ = qda_model.fit(X_train, y_train)

In [12]:
confusion(np.array(y_train), pd.Series(qda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	73	11
	0	87	129
Accuracy: 67.33%


In [13]:
confusion(np.array(y_val), pd.Series(qda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	34	8
	0	43	64
Accuracy: 65.77%


QDA can be regularized with a parameter between 0 and 1, so we can apply cross-validation in an attempt to obtain better metrics. In general, a small value of this regularization parameter (between 0.01 and 0.1) is desirable, but it does not improve results by much.

In [14]:
QDA_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', QuadraticDiscriminantAnalysis(priors = (0.5, 0.5)))])

n = 10
regs = np.logspace(start = -4, stop = -0.5, num = 100)

QDA_search = GridSearchCV(estimator = QDA_pipeline,
                          param_grid = {'QDA__reg_param' : regs},
                          scoring = 'f1_macro',
                          cv = 5)
QDA_search.fit(X, y)
QDA_search.best_params_

{'QDA__reg_param': 9.999999999999999e-05}

In [15]:
QDA_search.best_score_

0.6271432896385213

In [16]:
QDA_reg_param_ = QDA_search.best_params_['QDA__reg_param']
qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5),
                                          reg_param = QDA_reg_param_)
qda_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', qda_model)])

cross_val_results = pd.DataFrame(cross_validate(qda_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["QDA", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.648262,0.714125,0.674174,0.661648
QDA,0.627143,0.701966,0.665863,0.637054


## Naive Bayes



In [17]:
from sklearn.naive_bayes import GaussianNB

gaussian_nb = GaussianNB(priors = (0.5, 0.5))
gaussian_nb.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(gaussian_nb.predict(X_train)))

		Predicted
		+1	0
Real	+1	72	12
	0	93	123
Accuracy: 65.00%


In [18]:
confusion(np.array(y_val), pd.Series(gaussian_nb.predict(X_val)))

		Predicted
		+1	0
Real	+1	33	9
	0	46	61
Accuracy: 63.09%


In [19]:
gaussian_nb = GaussianNB(priors = (0.5, 0.5))
NB_pipeline = Pipeline([('scaler', StandardScaler()), ('NB', gaussian_nb)])

cross_val_results = pd.DataFrame(cross_validate(NB_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["GaussianNB", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.648262,0.714125,0.674174,0.661648
GaussianNB,0.6398,0.713779,0.674974,0.650412
QDA,0.627143,0.701966,0.665863,0.637054


## Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

logreg_model = LogisticRegression(C = 20, random_state = 42, class_weight = "balanced")

logreg_model.fit(X_train, y_train)
confusion(np.array(y_train), pd.Series(logreg_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	71	13
	0	77	139
Accuracy: 70.00%


/home/simple/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
confusion(np.array(y_val), pd.Series(logreg_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	33	9
	0	36	71
Accuracy: 69.80%


In [22]:
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight = "balanced"))])

n = 100
m = 10
Cs = np.logspace(start = -4, stop = 2, num = n)

logreg_search = GridSearchCV(estimator = logreg_pipeline,
                             param_grid = {'logreg__C' : Cs},
                             scoring = 'f1_macro',
                             cv = 5)
logreg_search.fit(X, y)
logreg_search.best_params_

{'logreg__C': 0.6579332246575682}

In [23]:
logreg_search.best_score_

0.6635322595910054

In [24]:
logreg_C = logreg_search.best_params_["logreg__C"]

logreg_model_best = LogisticRegression(C = logreg_C,
                                       class_weight = "balanced")
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', logreg_model_best)])

cross_val_results = pd.DataFrame(cross_validate(logreg_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["LogReg-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.663532,0.724197,0.681851,0.679451
LDA,0.648262,0.714125,0.674174,0.661648
GaussianNB,0.6398,0.713779,0.674974,0.650412
QDA,0.627143,0.701966,0.665863,0.637054


## Trying our best classifiers on our test data

In [25]:
test_data = pd.read_csv("minimal_test_fs.csv", header = None)
test_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female
0,11,-0.356675,6.383507,3.367296,4.2,1.40,0.142857,0
1,62,0.587787,5.411646,5.043425,4.0,0.80,0.500000,0
2,60,-0.356675,5.159055,2.639057,4.2,1.10,0.285714,0
3,60,1.740466,5.365976,6.745236,3.2,0.78,0.491228,1
4,48,-0.105361,5.164786,3.988984,2.7,0.90,0.222222,1


In [26]:
test_y = pd.read_csv("test_y.csv").iloc[:, 1]
test_y

0      1
1      0
2      1
3      0
4      1
      ..
111    1
112    0
113    1
114    0
115    0
Name: Label, Length: 116, dtype: int64

### LDA

In [27]:
lda_pipeline.fit(X, y)

labels_lda = pd.DataFrame(columns = ['ID', 'Label'])
labels_lda['Label'] = pd.DataFrame(lda_pipeline.predict(test_data))
labels_lda['ID'] = labels_lda.index + 1
labels_lda.to_csv('new_predictions/lda_best_fs.csv', index = False)
labels_lda

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,0
112,113,0
113,114,1
114,115,1


In [30]:
compute_metrics(test_y, labels_lda['Label'])

[0.6858872252309652,
 0.7312887915297553,
 0.6897274633123689,
 0.7068965517241379]

### QDA

In [31]:
qda_pipeline.fit(X, y)

labels_qda = pd.DataFrame(columns = ['ID', 'Label'])
labels_qda['Label'] = pd.DataFrame(qda_pipeline.predict(test_data))
labels_qda['ID'] = labels_qda.index + 1
labels_qda.to_csv('new_predictions/qda_best_fs.csv', index = False)
labels_qda

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,0


In [32]:
compute_metrics(test_y, labels_qda['Label'])

[0.6704545454545454, 0.719240598758671, 0.6789865871833085, 0.6896551724137931]

### Naive Bayes

In [33]:
NB_pipeline.fit(X, y)

labels_nb = pd.DataFrame(columns = ['ID', 'Label'])
labels_nb['Label'] = pd.DataFrame(NB_pipeline.predict(test_data))
labels_nb['ID'] = labels_nb.index + 1
labels_nb.to_csv('new_predictions/nb_best_fs.csv', index = False)
labels_nb

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,0


In [34]:
compute_metrics(test_y, labels_nb['Label'])

[0.6338383838383839,
 0.6768893756845564,
 0.6444113263785395,
 0.6551724137931034]

### Logistic Regression

In [35]:
logreg_pipeline.fit(X, y)

labels_logreg = pd.DataFrame(columns = ['ID', 'Label'])
labels_logreg['Label'] = pd.DataFrame(logreg_pipeline.predict(test_data))
labels_logreg['ID'] = labels_logreg.index + 1
labels_logreg.to_csv('new_predictions/logreg_best_fs.csv', index = False)
labels_logreg

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


In [36]:
compute_metrics(test_y, labels_logreg['Label'])

[0.6732394366197183,
 0.7283680175246441,
 0.6859946476360392,
 0.6896551724137931]

### Discrepancies

In [41]:
np.logical_and(labels_qda == labels_lda, labels_qda == labels_logreg).value_counts()

ID    Label
True  True     98
      False    18
Name: count, dtype: int64

In [43]:
(labels_qda == labels_logreg).value_counts()

ID    Label
True  True     102
      False     14
Name: count, dtype: int64